In [1]:
import pandas as pd, os, datetime

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')

nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess'
ehf_fpath = '/scratch/ng72/ms5578/time_series'
gen_fpath = '/scratch/ng72/ms5578/time_series/generation_18_19'

In [3]:
gen_details = pd.read_csv(f"{nmap_path}/nmap.csv")

hw_tseries = pd.read_csv(f"{ehf_fpath}/gen_hw_status.csv")
hw_tseries['time'] = pd.to_datetime(hw_tseries['time'])

For this example, I'll focus on plotting a 12 day period from 2018/11/23 to 2018/12/12.
No generators are in heatwave conditions from 11/09 to 11/22

In [4]:
hw_tseries = hw_tseries.set_index(['time']).sort_index()
hw_tseries = hw_tseries.loc['2018-11-09':'2018-12-12']
hw_tseries = hw_tseries.reset_index().set_index(['DUID','time']).sort_index()

Separating the states

In [5]:
def process_group(grp, gen_fpath, hw_tseries, start_date='2018-11-09', end_date='2018-12-12'):
    """
    Processes a single group of generator data.

    Parameters:
        grp (pd.DataFrame): A single group from gen_details (e.g., from groupby('region')).
        gen_fpath (str): Path to directory containing CSV files named by DUID (e.g., DUID.csv).
        hw_tseries (pd.DataFrame): DataFrame with columns 'time', 'DUID', and data to merge.
        start_date (str): Start date for subsetting the time series.
        end_date (str): End date for subsetting the time series.

    Returns:
        pd.DataFrame: The merged result for the group.
    """
    gen_locs = gen_fpath + '/' + grp['duid'] + ".csv"
    dfs = [pd.read_csv(fp) for fp in gen_locs if os.path.exists(fp)]

    if not dfs:
        return None

    dfs = pd.concat(dfs, ignore_index=True)

    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()

    dfs = dfs.loc[start_date:end_date]
    grouper = dfs.groupby(['DUID', pd.Grouper(freq='1h')])
    result = grouper['TOTALMWh'].sum().to_frame().reset_index()

    merged = pd.merge_asof(
        result.sort_values(by=['time', 'DUID']),
        hw_tseries.sort_values(by=['time', 'DUID']),
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )

    return merged


In [6]:
def process_all(gen_details, gen_fpath, hw_tseries, start_date='2018-11-09', end_date='2018-12-12'):
    """
    Process all generator data in gen_details without grouping.
    
    Parameters:
        gen_details (pd.DataFrame): DataFrame containing at least 'duid' column.
        gen_fpath (str): Directory path containing CSV files named by DUID.
        hw_tseries (pd.DataFrame): Heatwave timeseries data with 'time' and 'DUID'.
        start_date (str): Start date for filtering time series.
        end_date (str): End date for filtering time series.
    
    Returns:
        pd.DataFrame: Merged dataframe with summed TOTALMWh per DUID per hour and heatwave info.
    """
    # Build file paths for all DUIDs
    gen_locs = gen_fpath + '/' + gen_details['duid'] + ".csv"

    # Read all existing CSVs
    dfs = [pd.read_csv(fp) for fp in gen_locs if os.path.exists(fp)]
    
    if not dfs:
        print("[WARN] No data files found for any DUID.")
        return pd.DataFrame()  # Return empty if no files
    
    # Concatenate all data
    dfs = pd.concat(dfs, ignore_index=True)

    # Convert time to datetime and filter
    dfs['time'] = pd.to_datetime(dfs['time'])
    dfs = dfs.set_index('time').sort_index()
    dfs = dfs.loc[start_date:end_date]

    # Group by DUID and hourly time, sum TOTALMWh
    grouped = dfs.groupby(['DUID', pd.Grouper(freq='1h')])['TOTALMWh'].sum().reset_index()

    # Sort hw_tseries once
    hw_sorted = hw_tseries.sort_values(by=['time', 'DUID'])

    # Merge with heatwave timeseries
    merged = pd.merge_asof(
        grouped.sort_values(by=['time', 'DUID']),
        hw_sorted,
        by='DUID',
        on='time',
        tolerance=pd.Timedelta("1d"),
        direction='nearest'
    )
    
    return merged

In [7]:
# st_group = gen_details.groupby('region')
# states = {region: process_group(grp, gen_fpath, hw_tseries) for region, grp in st_group}

# qld_df,nsw_df,vic_df,sa_df,tas_df = states['QLD1'],states['NSW1'],states['VIC1'],states['SA1'],states['TAS1']

In [20]:
df = process_all(gen_details, gen_fpath, hw_tseries)
df = df.merge(gen_details[['duid', 'region','fuel_source_primary']], left_on='DUID', right_on='duid', how='left').drop(columns='duid')

Using KSP1 as the first generator in the heatwave for QLD and SUNRSF1 as first for VIC, STWF1 (NSW) has next longest hw of 5 days

In [21]:
def highLights(df, fig, variable, level, mode, fillcolor, layer):
    """
    Set a specified color as background for given
    levels of a specified variable using a shape.
    
    Keyword arguments:
    ==================
    fig -- plotly figure
    variable -- column name in a pandas dataframe
    level -- int or float
    mode -- set threshold above or below
    fillcolor -- any color type that plotly can handle
    layer -- position of shape in plotly fiugre, like "below"
    
    """
    
    if mode == 'above':
        m = df[variable].gt(level)
    
    if mode == 'below':
        m = df[variable].lt(level)
        
    df1 = df[m].groupby((~m).cumsum())['time'].agg(['first','last'])

    for index, row in df1.iterrows():
        #print(row['first'], row['last'])
        fig.add_shape(type="rect",
                        xref="x",
                        yref="paper",
                        x0=row['first'],
                        y0=0,
                        x1=row['last'],
                        y1=1,
                        line=dict(color="rgba(0,0,0,0)",width=3,),
                        fillcolor=fillcolor,
                        layer=layer) 
    return(fig)


In [22]:
# # Plot a single generator
# test = nsw_df[nsw_df['DUID']== 'STWF1']

def plot_unit(df):
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(x=df['time'], y=df['TOTALMWh']))
    
    # Set title
    fig.update_layout(
        title_text="Time series with range slider and selectors"
    )
    
    # Add range slider
    fig.update_layout(
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=7,
                         label="1w",
                         step="day",
                         stepmode="backward"),
                    dict(count=1,
                         label="1m",
                         step="month",
                         stepmode="backward"),
                    dict(step="all")
                ])
            ),
            rangeslider=dict(
                visible=True
            ),
            type="date"
        )
    )
    
    fig = highLights(df=df, fig = fig, variable = 'EHF_flag', level = 0, mode = 'above',
                   fillcolor = 'rgba(200,0,200,0.2)', layer = 'below')
    
    return fig

# fig = plot_unit(test)
# fig.show()

In [23]:
def plot_agg_group(grouped_df, group_col, title='Time Series Plot'):
    fig = go.Figure()

    for group_name, group in grouped_df.groupby(group_col):
        fig.add_trace(go.Scatter(
            x=group['time'],
            y=group['TOTALMWh'],
            mode='lines',
            name=str(group_name)
        ))

        fig = highLights(
            df=group,             # Only this group's data
            fig=fig,
            variable='EHF_flag',  # Column to check
            level=0,              # Threshold
            mode='above',         # or 'below'
            fillcolor='rgba(255,0,0,0.1)',  # Semi-transparent red
            layer='below'
        )

    fig.update_layout(
        title=title,
        xaxis_title='Time',
        yaxis_title='Total MWh',
        template='plotly_white',
        legend_title=group_col
    )

    fig.show()

In [24]:
agg_func = {'TOTALMWh':'sum','EHF_flag':'max'}
reg_grp = df.groupby(['region', 'time']).aggregate(agg_func).reset_index()
fuel_grp = df.groupby(['fuel_source_primary', 'time']).aggregate(agg_func).reset_index()

# reg_grp['normalised'] = df.groupby('region')['TOTALMWh'].transform(
#                         lambda x: (x - x.min()) / (x.max() - x.min()))

plot_agg_group(reg_grp, group_col='region', title='Total MWh by Region')
plot_agg_group(fuel_grp, group_col='fuel_source_primary', title='Total MWh by Fuel Source')

KeyError: 'normalised'